In [ ]:
"""
Simple RAG Application
- Embeddings: sentence-transformers (free, local)
- Vector Store: FAISS (free, local)
- LLM: Groq API (free tier) -- swap with Ollama for fully local
"""

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from groq import Groq

# ── Config ───────────────────────────────────────────────────────────────────
GROQ_API_KEY = "gsk_key"  # get free key at console.groq.com
EMBED_MODEL  = "all-MiniLM-L6-v2"        # fast, free, local
LLM_MODEL    = "llama3-8b-8192"          # free on Groq

# ── Embedder + Index ─────────────────────────────────────────────────────────
embedder = SentenceTransformer(EMBED_MODEL)

class RAG:
    def __init__(self):
        self.docs   = []
        self.index  = None
        self.client = Groq(api_key=GROQ_API_KEY)

    def add_documents(self, documents: list[str]):
        """Embed and index a list of text chunks."""
        self.docs = documents
        vecs = embedder.encode(documents, convert_to_numpy=True)
        dim  = vecs.shape[1]
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(vecs.astype("float32"))
        print(f"Indexed {len(documents)} documents.")

    def retrieve(self, query: str, top_k: int = 3) -> list[str]:
        """Return top_k most relevant chunks for the query."""
        q_vec = embedder.encode([query], convert_to_numpy=True).astype("float32")
        _, indices = self.index.search(q_vec, top_k)
        return [self.docs[i] for i in indices[0] if i < len(self.docs)]

    def ask(self, question: str, top_k: int = 3) -> str:
        """Retrieve context then ask the LLM."""
        context_chunks = self.retrieve(question, top_k)
        context = "\n\n".join(context_chunks)

        prompt = f"""Use the context below to answer the question.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {question}
Answer:"""

        response = self.client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content


c:\Users\shant\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shant\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
